In [1]:
# Install Dependencies
%pip install -q --upgrade numerapi numerai-tools optuna seaborn

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
import random
from datetime import timedelta
import time
import warnings
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass
import gc
from functools import partial

# Numerai
from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

# Files
import json
import os
import pickle
import cloudpickle
import shutil
from tqdm import tqdm

# ML
import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer

import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour
)

# Display Settings
pd.set_option("display.max_columns", 500)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Inline plots
%matplotlib inline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 14.5 MB/s eta 0:00:00


In [2]:
with open ('/content/drive/MyDrive/Colab Notebooks/Step_1/fm1.cpkl', 'rb') as f:
    FM1 = cloudpickle.load(f)

with open('/content/drive/MyDrive/Colab Notebooks/Step_1/grouped_set_medium.pkl', 'rb') as f:
    dict_medium = pickle.load(f)

feature_set = list(dict_medium['all'])

fncv3_features = list(dict_medium['fncv3_features'])

exclude_groups = {'intelligence', 'charisma', 'v2_equivalent_features', 'v3_equivalent_features'}

other_features = []
for key, features in dict_medium.items():
    if key not in exclude_groups and key != 'fncv3_features' and key != 'all':
        other_features.extend(features)

other_features = [f for f in set(other_features) if f not in fncv3_features]

In [3]:
napi = NumerAPI()

all_datasets = napi.list_datasets()
dataset_versions = list(set(d.split('/')[0] for d in all_datasets))
DATA_VERSION = max(dataset_versions)

napi.download_dataset(f"{DATA_VERSION}/live.parquet")
napi.download_dataset(f"{DATA_VERSION}/live_benchmark_models.parquet")

live_features = pd.read_parquet(f"{DATA_VERSION}/live.parquet", columns=feature_set)
live_benchmark = pd.read_parquet(f"{DATA_VERSION}/live_benchmark_models.parquet")

In [4]:
def get_preds_neut(live_features: pd.DataFrame) -> pd.DataFrame:

    preds_neut = pd.DataFrame(FM1.predict(live_features[feature_set]), index=live_features.index, columns=['prediction'])

    preds_neut = neutralize(preds_neut, live_features[fncv3_features], proportion=0.5)
    preds_neut = neutralize(preds_neut, live_features[other_features], proportion=0.15)

    return preds_neut.rank(pct=True)

preds_neut = get_preds_neut(live_features)
preds_neut.head(5)


,prediction
id,
n000024cd16076a0,0.532893
n0003b5d5c990c24,0.991439
n000f16b5f9c835e,0.770051
n0011cf18c435a37,0.308501
n0013b5169396526,0.397266


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(preds_neut['prediction'], bins=100, alpha=0.7, color='skyblue', edgecolor='black')
plt.title('Распределение предсказаний после rank(pct=True)')
plt.xlabel('Prediction (percentile)')
plt.ylabel('Частота')
plt.xlim(0, 1)
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
FM1 = cloudpickle.dumps(get_preds_neut)
with open("/content/drive/MyDrive/Colab Notebooks/Step_1/FM1.pkl", "wb") as f:
    f.write(FM1)



In [2]:
with open("/content/drive/MyDrive/Colab Notebooks/Step_1/FM1.pkl", "rb") as f:
    test = cloudpickle.load(f)
type(test)

function

In [3]:
test.__name__

'get_preds_neut'

In [ ]:


# Generate live predictions
live_predictions = model.predict(live_features[feature_set])

# Format submission
pd.Series(live_predictions, index=live_features.index).to_frame("prediction")

# Define your prediction pipeline as a function
def predict(live_features: pd.DataFrame, _live_benchmark_models: pd.DataFrame) -> pd.DataFrame:
    live_predictions = model.predict(live_features[feature_set])
    submission = pd.Series(live_predictions, index=live_features.index)
    return submission.to_frame("prediction")

# Use the cloudpickle library to serialize your function
import cloudpickle
p = cloudpickle.dumps(predict)
with open("hello_numerai.pkl", "wb") as f:
    f.write(p)